## Resource requests and limits

Every container can declare what it expects — two optional, per-container fields:

```yaml
resources:
  requests: { cpu: 250m, memory: 256Mi }   # reserve this much
  limits:   { cpu: 500m, memory: 512Mi }   # never exceed this
```

**`requests` = the minimum the scheduler must reserve.** It sums the requests of all pods on a node, subtracts from allocatable capacity, and only places a pod if its requests fit the headroom. It is a **scheduling input, not a runtime cap** — a container can use *more* if the node has it free.

**`limits` = the maximum the runtime allows**, enforced by the kernel cgroup:

- **CPU over limit** → the container is **throttled** (slowed, no restart).
- **Memory over limit** → the kernel **OOM-kills** the process; the container restarts per `restartPolicy`.

**Units:** CPU `1` = one core, `500m` = half a core (millicores). Memory in bytes with binary suffixes `Ki/Mi/Gi/Ti`.

**Omissions matter:**

- requests omitted, limits set → requests **default to limits**.
- requests set, limits omitted → **no upper bound**, can hog the node.
- both omitted → schedules anywhere, runs **unbounded** — dangerous in shared clusters (fix with `LimitRange`).

There's also **`ephemeral-storage`** (node scratch: `emptyDir`, writable layer, logs) — same request/limit shape; the kubelet evicts over-users. On our map these are the **requests** and **limits** chips in the Scheduling box — the numbers the scheduler reserves and the kernel enforces.